In [2]:
%pip install xgboost scikit-learn matplotlib seaborn

  Using cached xgboost-3.3.0-py3-none-win_amd64.whl.metadata (2.0 kB)
Using cached xgboost-3.3.0-py3-none-win_amd64.whl (69.5 MB)
Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.metrics import roc_auc_score, classification_report

# Set visual style
sns.set_theme(style="whitegrid")

# 1. Generate Realistic Synthetic Inventory Data (No Direct Formula Leakage)
np.random.seed(42)
n_samples = 1000

# Feature generation
sales_velocity_daily = np.random.gamma(shape=2.0, scale=10.0, size=n_samples) + 1.0
current_stock = sales_velocity_daily * np.random.uniform(2, 35, size=n_samples)
supplier_lead_time = np.random.normal(loc=12, scale=4, size=n_samples).clip(1)
lead_time_variance = np.random.exponential(scale=2.5, size=n_samples)
holding_days = np.random.exponential(scale=30, size=n_samples)
unit_price = np.random.uniform(15, 250, size=n_samples)

data = pd.DataFrame({
    'sku_id': [f'SKU_{i:04d}' for i in range(n_samples)],
    'current_stock': current_stock,
    'sales_velocity_30d': sales_velocity_daily * 30,
    'supplier_lead_time_days': supplier_lead_time,
    'lead_time_variance': lead_time_variance,
    'holding_days': holding_days,
    'price_per_unit': unit_price
})

# Feature Engineering
data['days_of_supply'] = data['current_stock'] / (data['sales_velocity_30d'] / 30 + 1e-5)
data['capital_tied_up'] = data['current_stock'] * data['price_per_unit']

# 2. Complex Log-Odds Function with Heavy Stochastic Supply Chain Noise
# Balanced baseline (-0.8 centering offset)
log_odds = (
    -0.03 * data['days_of_supply']
    + 0.04 * data['supplier_lead_time_days']
    + 0.12 * data['lead_time_variance']
    + 0.015 * data['holding_days']
    - 0.8
)

# Convert to probabilities with significant unobserved real-world variance (e.g. demand surges, transport delays)
probabilities = 1 / (1 + np.exp(-log_odds))
stochastic_noise = np.random.logistic(loc=0, scale=0.45, size=n_samples)
prob_with_noise = 1 / (1 + np.exp(-(log_odds + stochastic_noise)))

# Binary target (Balanced ~35-40% positive instances)
data['is_misaligned'] = (prob_with_noise > 0.50).astype(int)

print("--- 1. DATASET OVERVIEW ---")
print(data[['sku_id', 'current_stock', 'days_of_supply', 'holding_days', 'is_misaligned']].head())
print("\nBalanced Class Distribution:")
print(data['is_misaligned'].value_counts())

# 3. Train / Test Split
X = data[['current_stock', 'sales_velocity_30d', 'supplier_lead_time_days', 
          'lead_time_variance', 'holding_days', 'days_of_supply', 'capital_tied_up']]
y = data['is_misaligned']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Model Benchmarking Pipeline
# Note: Logistic Regression gets StandardScaler to resolve scaling/convergence issues
models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42),
    "XGBoost": xgb.XGBClassifier(n_estimators=80, max_depth=3, learning_rate=0.03, eval_metric='logloss', random_state=42)
}

print("\n--- 2. REALISTIC BENCHMARK SCORES ---")
for name, model in models.items():
    model.fit(X_train, y_train)
    probs = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, probs)
    print(f"[{name}] ROC-AUC Score: {auc:.4f}")

# Save the calibrated XGBoost model
best_xgb_model = models["XGBoost"]
best_xgb_model.save_model("../models/xgboost_stockout.json")
print("\nSuccessfully saved calibrated XGBoost model to ../models/xgboost_stockout.json!")

--- 1. DATASET OVERVIEW ---
     sku_id  current_stock  days_of_supply  holding_days  is_misaligned
0  SKU_0000     446.552019       17.907348      9.035100              1
1  SKU_0001     135.070687        8.471219     14.586822              0
2  SKU_0002     328.410932       22.155727     85.932061              1
3  SKU_0003     166.944979       11.262539     28.277198              0
4  SKU_0004     419.435647        8.830753     54.049528              1

Balanced Class Distribution:
is_misaligned
0    572
1    428
Name: count, dtype: int64

--- 2. REALISTIC BENCHMARK SCORES ---
[Logistic Regression] ROC-AUC Score: 0.8255
[Random Forest] ROC-AUC Score: 0.8047
[XGBoost] ROC-AUC Score: 0.8120

Successfully saved calibrated XGBoost model to ../models/xgboost_stockout.json!


In [6]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# 1. Target: Continuous Dollar Value at Risk (Holding Penalties / Capital Tied Up)
data['holding_penalty_usd'] = data['capital_tied_up'] * 0.02 * (data['holding_days'] / 30)

X_reg = data[['current_stock', 'sales_velocity_30d', 'supplier_lead_time_days', 'holding_days']]
y_reg = data['holding_penalty_usd']

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# 2. Fit Linear Regression Model
lin_reg = LinearRegression()
lin_reg.fit(X_train_r, y_train_r)

y_pred_r = lin_reg.predict(X_test_r)

print("--- LINEAR REGRESSION (COST PREDICTION) ---")
print(f"R² Score (Variance Explained): {r2_score(y_test_r, y_pred_r):.4f}")
print(f"Mean Absolute Error: ${np.mean(np.abs(y_test_r - y_pred_r)):.2f}")

--- LINEAR REGRESSION (COST PREDICTION) ---
R² Score (Variance Explained): 0.5136
Mean Absolute Error: $783.87


In [7]:
# BUSINESS ROI QUANTIFICATION 
# Translating XGBoost predictions into quantifiable financial impact

# Reconstruct the test set with predictions
impact_df = X_test.copy()
impact_df['actual_misalignment'] = y_test
impact_df['predicted_prob'] = best_xgb_model.predict_proba(X_test)[:, 1]
# Flag items where the model predicts > 50% risk
impact_df['predicted_misalignment'] = (impact_df['predicted_prob'] > 0.50).astype(int)

# --- Define the Business Rules for Financial Leakage ---

# 1. Deadstock Penalty: 2% monthly holding cost for capital tied up
impact_df['deadstock_penalty_risk'] = impact_df['capital_tied_up'] * 0.02 * (impact_df['holding_days'] / 30)

# 2. Stockout Penalty: 15% premium paid for express freight to save a client relationship
impact_df['express_freight_risk'] = impact_df['capital_tied_up'] * 0.15

# Total Risk exposure assumes the worst of the two penalties applies per misaligned SKU
impact_df['total_financial_risk'] = impact_df[['deadstock_penalty_risk', 'express_freight_risk']].max(axis=1)

# --- Calculate Model ROI ---

# Baseline: Total money the business is losing if they do nothing (Actual Positives)
total_leakage = impact_df.loc[impact_df['actual_misalignment'] == 1, 'total_financial_risk'].sum()

# Model Intervention: Money saved because the model successfully predicted it (True Positives)
# (Action taken 30 days in advance prevents the penalty)
true_positives = impact_df[(impact_df['actual_misalignment'] == 1) & (impact_df['predicted_misalignment'] == 1)]
dollars_saved = true_positives['total_financial_risk'].sum()

# Missed Opportunity: Money lost because the model failed to predict it (False Negatives)
false_negatives = impact_df[(impact_df['actual_misalignment'] == 1) & (impact_df['predicted_misalignment'] == 0)]
dollars_lost = false_negatives['total_financial_risk'].sum()

# Print the Executive Dashboard Output
print("--- 3. EXECUTIVE ROI & FINANCIAL IMPACT SUMMARY ---")
print(f"Total Profit Leakage at Risk (Test Set):     ${total_leakage:,.2f}")
print(f"Leakage Prevented by ML Model (Saved):       ${dollars_saved:,.2f}")
print(f"Leakage Missed by Model (Lost):              ${dollars_lost:,.2f}")
print("-" * 55)
print(f"Model ROI Capture Rate:                      {(dollars_saved / total_leakage) * 100:.1f}%")
print(f"Annualized Projected Savings (x12 months):   ${(dollars_saved * 12):,.2f}")

--- 3. EXECUTIVE ROI & FINANCIAL IMPACT SUMMARY ---
Total Profit Leakage at Risk (Test Set):     $555,689.42
Leakage Prevented by ML Model (Saved):       $326,418.69
Leakage Missed by Model (Lost):              $229,270.73
-------------------------------------------------------
Model ROI Capture Rate:                      58.7%
Annualized Projected Savings (x12 months):   $3,917,024.30
